# System-vector test — NumPy vs Torch backends

Goal of this notebook: check that the **whole system as one `(N,3)` array** works,
and that the *same* force kernel runs on either backend by swapping the array module `xp`.
No ECS, no force-map, no rendering here — just the vector approach and timing.

## 1. Load the system into Structure-of-Arrays (backend-neutral)

In [ ]:
import numpy as np
import markup_manager as mm

G = 0.0001184069  # same constant as mymath.py

def get_bodies(path):
    result = mm.get_toml(path)["Bodies"]
    objects = []
    for key, obj in result.items():
        obj["Name"] = key
        objects.append(obj)
    return objects

objects = get_bodies("system.toml")

# One row per body. Particle i is row i in every array — nothing per-particle is lost.
pos0  = np.array([o["R (polar)"] for o in objects], dtype=np.float64)  # (N,3)
vel0  = np.array([o["V (polar)"] for o in objects], dtype=np.float64)  # (N,3)
mass0 = np.array([o["Mass"]      for o in objects], dtype=np.float64)  # (N,)
names = [o["Name"] for o in objects]

N = len(objects)
print(f"N = {N}")
print("bodies:", names)
print("pos0 shape:", pos0.shape)

N = 10
bodies: ['Sun', 'Earth', 'Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Moon']
pos0 shape: (10, 3)


## 2. One kernel, written only in terms of `xp` (works for numpy OR torch)

`accel` is the vectorised replacement for the whole triple-nested loop.
Self-interaction (`i==j`) vanishes automatically: there `d=0`, so the term is `0`.
Softening `eps` keeps close approaches finite (this is the C5 fix).

In [ ]:
def accel(xp, pos, mass, G, eps=1e-3):
    d   = pos[None, :, :] - pos[:, None, :]   # (N,N,3): d[i,j] = r_j - r_i
    r2  = (d ** 2).sum(axis=-1) + eps ** 2    # (N,N)
    inv = r2 ** -1.5                          # (N,N)
    return G * (inv[:, :, None] * d * mass[None, :, None]).sum(axis=1)  # (N,3)

def run(xp, pos, vel, mass, n_steps, h, G):
    """Semi-implicit Euler. State stays as (N,3) arrays the whole time —
    no growing history, nothing leaves the backend."""
    for _ in range(n_steps):
        a   = accel(xp, pos, mass, G)
        vel = vel + h * a
        pos = pos + h * vel
    return pos, vel

In [ ]:
h       = 1e-4
t_end   = 1.0
n_steps = int(t_end / h)
print(f"n_steps = {n_steps}")

n_steps = 10000


## 3. NumPy backend (CPU)

In [ ]:
from time import perf_counter

pos_np = pos0.copy()
vel_np = vel0.copy()
mass_np = mass0.copy()

t0 = perf_counter()
pos_np, vel_np = run(np, pos_np, vel_np, mass_np, n_steps, h, G)
t_numpy = perf_counter() - t0

print(f"numpy: {t_numpy:.3f} s  ({n_steps/t_numpy:,.0f} steps/s)")
print("final Earth pos:", pos_np[names.index('Earth')])

numpy: 0.087 s  (115,378 steps/s)
final Earth pos: [ 1.01729299 -0.01513605  0.        ]


## 4. Torch backend (GPU if available)

Same `run` / `accel` — only the array construction and device change.
Data is created on the device **once** and stays there for the whole loop.
We synchronise only to time it (and would copy to CPU only to draw/log).

In [ ]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
print("device:", device)

def sync():
    if device.type == "mps":  torch.mps.synchronize()
    elif device.type == "cuda": torch.cuda.synchronize()

# build on device ONCE
pos_t  = torch.tensor(pos0,  dtype=torch.float32, device=device)
vel_t  = torch.tensor(vel0,  dtype=torch.float32, device=device)
mass_t = torch.tensor(mass0, dtype=torch.float32, device=device)

sync()
t0 = perf_counter()
pos_t, vel_t = run(torch, pos_t, vel_t, mass_t, n_steps, h, G)
sync()                       # wait for the GPU before stopping the clock
t_torch = perf_counter() - t0

print(f"torch: {t_torch:.3f} s  ({n_steps/t_torch:,.0f} steps/s)")
print("final Earth pos:", pos_t[names.index('Earth')].cpu().numpy())

device: mps
torch: 1.312 s  (7,622 steps/s)
final Earth pos: [ 1.0172839  -0.01506162  0.        ]


## 5. Compare

Expect: at small N the **CPU/NumPy wins** — kernel-launch overhead dominates tiny arrays.
The GPU only pays off once N is large (next cell).

In [ ]:
diff = np.abs(pos_np - pos_t.cpu().numpy()).max()
print(f"numpy : {t_numpy:.3f} s")
print(f"torch : {t_torch:.3f} s")
print(f"max |pos_numpy - pos_torch| = {diff:.2e}   (float32 vs float64, so ~1e-3 is fine)")

numpy : 0.087 s
torch : 1.312 s
max |pos_numpy - pos_torch| = 2.27e-04   (float32 vs float64, so ~1e-3 is fine)


## 6. Where the GPU wins: large N

Same kernel, N=2000 random bodies, a few steps. This is the crossover the
per-vector version could never reach.

In [ ]:
Nbig, steps_big = 2000, 20
rng = np.random.default_rng(0)
p = rng.normal(0, 10, (Nbig, 3)); v = np.zeros((Nbig, 3)); m = rng.uniform(0.1, 1, Nbig)

t0 = perf_counter(); run(np, p.copy(), v.copy(), m, steps_big, h, G); t_np_big = perf_counter() - t0

pt = torch.tensor(p, dtype=torch.float32, device=device)
vt = torch.tensor(v, dtype=torch.float32, device=device)
mt = torch.tensor(m, dtype=torch.float32, device=device)
sync(); t0 = perf_counter(); run(torch, pt, vt, mt, steps_big, h, G); sync(); t_t_big = perf_counter() - t0

print(f"N={Nbig}:  numpy {t_np_big:.3f} s   torch {t_t_big:.3f} s   speedup x{t_np_big/t_t_big:.1f}")

N=2000:  numpy 1.948 s   torch 0.323 s   speedup x6.0
